# Dataset Cleaning

### Importing all the necessary libraries

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

### Setting up file paths

In [4]:
# This notebook is inside the same Dataset folder so I'll use:

DATA_DIR = Path.cwd()

# I'll save all the clean .csv files in a new file:

OUTPUT_DIR = DATA_DIR / "cleaned_csv"
OUTPUT_DIR.mkdir(exist_ok=True)

# Let me give them short names for ease:

files = {
    "customers": DATA_DIR / "semicon_customers.csv",
    "date": DATA_DIR / "semicon_date.csv",
    "markets": DATA_DIR / "semicon_markets.csv",
    "products": DATA_DIR / "semicon_products_named.csv",
    "transactions": DATA_DIR / "semicon_transactions.csv",
}

### Loading the files

#### **customers**

In [9]:
customers = pd.read_csv(files["customers"])
customers.head()

,customer_code,custmer_name,customer_type
0,Cus001,Surge Stores,Brick & Mortar
1,Cus002,Nomad Stores,Brick & Mortar
2,Cus003,Excel Stores,Brick & Mortar
3,Cus004,Surface Stores,Brick & Mortar
4,Cus005,Premium Stores,Brick & Mortar


In [10]:
# Understanding customers table structure

customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_code  38 non-null     object
 1   custmer_name   38 non-null     object
 2   customer_type  38 non-null     object
dtypes: object(3)
memory usage: 1.0+ KB


#### **date**

In [11]:
date_dim = pd.read_csv(files["date"])
date_dim.head()

,date,cy_date,year,month_name,date_yy_mmm
0,2017-06-01,2017-06-01,2017,June,17-Jun\r
1,2017-06-02,2017-06-01,2017,June,17-Jun\r
2,2017-06-03,2017-06-01,2017,June,17-Jun\r
3,2017-06-04,2017-06-01,2017,June,17-Jun\r
4,2017-06-05,2017-06-01,2017,June,17-Jun\r


In [12]:
# Understanding date_dim table structure

date_dim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1126 entries, 0 to 1125
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         1126 non-null   object
 1   cy_date      1126 non-null   object
 2   year         1126 non-null   int64 
 3   month_name   1126 non-null   object
 4   date_yy_mmm  1126 non-null   object
dtypes: int64(1), object(4)
memory usage: 44.1+ KB


#### **markets**

In [13]:
markets = pd.read_csv(files["markets"])
markets.head()

,markets_code,markets_name,zone
0,Mark001,Chennai,South
1,Mark002,Mumbai,Central
2,Mark003,Ahmedabad,North
3,Mark004,Delhi NCR,North
4,Mark005,Kanpur,North


In [14]:
# Understanding markets table structure

markets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   markets_code  17 non-null     object
 1   markets_name  17 non-null     object
 2   zone          15 non-null     object
dtypes: object(3)
memory usage: 540.0+ bytes


#### **products**

In [15]:
products = pd.read_csv(files["products"])
products.head()

,product_code,product_name,product_type
0,Prod001,Semicon Vortex V3-1000 Desktop CPU,Own Brand
1,Prod002,Semicon Vortex V3-1100 Desktop CPU,Own Brand
2,Prod003,Semicon Vortex V3-1200 Desktop CPU,Own Brand
3,Prod004,Semicon Vortex V3-1300 Desktop CPU,Own Brand
4,Prod005,Semicon Vortex V3-1400 Desktop CPU,Own Brand


In [16]:
# Understanding products table structure

products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_code  279 non-null    object
 1   product_name  279 non-null    object
 2   product_type  279 non-null    object
dtypes: object(3)
memory usage: 6.7+ KB


#### **transactions**

In [17]:
transactions = pd.read_csv(files["transactions"])
transactions.head()

,product_code,customer_code,market_code,order_date,sales_qty,sales_amount,currency
0,Prod001,Cus001,Mark001,2017-10-10,100,41241,INR
1,Prod001,Cus002,Mark002,2018-05-08,3,-1,INR
2,Prod002,Cus003,Mark003,2018-04-06,1,875,INR
3,Prod002,Cus003,Mark003,2018-04-11,1,583,INR
4,Prod002,Cus004,Mark003,2018-06-18,6,7176,INR


In [19]:
# Understanding transactions table structure

transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_code   50000 non-null  object
 1   customer_code  50000 non-null  object
 2   market_code    50000 non-null  object
 3   order_date     50000 non-null  object
 4   sales_qty      50000 non-null  int64 
 5   sales_amount   50000 non-null  int64 
 6   currency       50000 non-null  object
dtypes: int64(2), object(5)
memory usage: 2.7+ MB


### Creating some helper functions for ease & time-saving

In [20]:
# Function 1:

def clean_text_columns(df):
    """Strip whitespace and remove hidden carriage returns/newlines from text columns."""
    df = df.copy()
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace("\r", "", regex=False)
            .str.replace("\n", " ", regex=False)
            .str.strip()
        )
    
    return df

# Function 2:

def basic_profile(df, name):
    """Quick data quality profile."""
    print(f"\n{name}")
    print("-" * len(name))
    print("Shape:", df.shape)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nData types:")
    print(df.dtypes)

### Displaying a quick-profile of all tables

In [21]:
for name, df in {
    "customers": customers,
    "date_dim": date_dim,
    "markets": markets,
    "products": products,
    "transactions": transactions,
}.items():
    basic_profile(df, name)


customers
---------
Shape: (38, 3)

Missing values:
customer_code    0
custmer_name     0
customer_type    0
dtype: int64

Duplicate rows: 0

Data types:
customer_code    object
custmer_name     object
customer_type    object
dtype: object

date_dim
--------
Shape: (1126, 5)

Missing values:
date           0
cy_date        0
year           0
month_name     0
date_yy_mmm    0
dtype: int64

Duplicate rows: 0

Data types:
date           object
cy_date        object
year            int64
month_name     object
date_yy_mmm    object
dtype: object

markets
-------
Shape: (17, 3)

Missing values:
markets_code    0
markets_name    0
zone            2
dtype: int64

Duplicate rows: 0

Data types:
markets_code    object
markets_name    object
zone            object
dtype: object

products
--------
Shape: (279, 3)

Missing values:
product_code    0
product_name    0
product_type    0
dtype: int64

Duplicate rows: 0

Data types:
product_code    object
product_name    object
product_type    object
d

### Data cleaning for **customers** table

In [22]:
# Cleaning customers table with function:

customers_clean = clean_text_columns(customers)

# Fixing column typo:
customers_clean = customers_clean.rename(columns={
    "custmer_name": "customer_name"
})

# Removing exact duplicates, if any:
customers_clean = customers_clean.drop_duplicates().reset_index(drop=True)

# Validating important columns:
assert customers_clean["customer_code"].isna().sum() == 0
assert customers_clean["customer_code"].duplicated().sum() == 0

customers_clean.head()

,customer_code,customer_name,customer_type
0,Cus001,Surge Stores,Brick & Mortar
1,Cus002,Nomad Stores,Brick & Mortar
2,Cus003,Excel Stores,Brick & Mortar
3,Cus004,Surface Stores,Brick & Mortar
4,Cus005,Premium Stores,Brick & Mortar


### Data cleaning for **date_dim** table

In [23]:
# Cleaning date_dim table with function:

date_clean = clean_text_columns(date_dim)

# Converting date_dim columns to real datetime values:

date_clean["date"] = pd.to_datetime(date_clean["date"], errors="coerce")
date_clean["cy_date"] = pd.to_datetime(date_clean["cy_date"], errors="coerce")

# Rebuilding date helper columns to remove hidden characters and ensuring consistency:

date_clean["year"] = date_clean["date"].dt.year
date_clean["month_name"] = date_clean["date"].dt.month_name()
date_clean["cy_date"] = date_clean["date"].values.astype("datetime64[M]")
date_clean["date_yy_mmm"] = date_clean["date"].dt.strftime("%y-%b")

# Removing duplicates:

date_clean = date_clean.drop_duplicates().reset_index(drop=True)

# Validating:

assert date_clean["date"].isna().sum() == 0
assert date_clean["date"].duplicated().sum() == 0

date_clean.head()

,date,cy_date,year,month_name,date_yy_mmm
0,2017-06-01,2017-06-01,2017,June,17-Jun
1,2017-06-02,2017-06-01,2017,June,17-Jun
2,2017-06-03,2017-06-01,2017,June,17-Jun
3,2017-06-04,2017-06-01,2017,June,17-Jun
4,2017-06-05,2017-06-01,2017,June,17-Jun


### Data cleaning for **markets** table

In [24]:
# Cleaning markets table with function:

markets_clean = clean_text_columns(markets)

# Renaming to match transactions table

markets_clean = markets_clean.rename(columns={
    "markets_code": "market_code"
})

# Filling missing zones for international markets

markets_clean["zone"] = markets_clean["zone"].fillna("International")

# Removing duplicates

markets_clean = markets_clean.drop_duplicates().reset_index(drop=True)

# Validating

assert markets_clean["market_code"].isna().sum() == 0
assert markets_clean["market_code"].duplicated().sum() == 0

markets_clean.head()

,market_code,markets_name,zone
0,Mark001,Chennai,South
1,Mark002,Mumbai,Central
2,Mark003,Ahmedabad,North
3,Mark004,Delhi NCR,North
4,Mark005,Kanpur,North


### Data cleaning for **products** table

In [25]:
# Cleaning products table:

products_clean = clean_text_columns(products)

# Removing duplicates:

products_clean = products_clean.drop_duplicates().reset_index(drop=True)

# Validating:

assert products_clean["product_code"].isna().sum() == 0
assert products_clean["product_code"].duplicated().sum() == 0

products_clean.head()

,product_code,product_name,product_type
0,Prod001,Semicon Vortex V3-1000 Desktop CPU,Own Brand
1,Prod002,Semicon Vortex V3-1100 Desktop CPU,Own Brand
2,Prod003,Semicon Vortex V3-1200 Desktop CPU,Own Brand
3,Prod004,Semicon Vortex V3-1300 Desktop CPU,Own Brand
4,Prod005,Semicon Vortex V3-1400 Desktop CPU,Own Brand


### Data cleaning for **transactions** table

In [26]:
# Cleaning transactions table:

transactions_clean = clean_text_columns(transactions)

# Converting date and numeric columns:

transactions_clean["order_date"] = pd.to_datetime(
    transactions_clean["order_date"], 
    errors="coerce"
)

transactions_clean["sales_qty"] = pd.to_numeric(
    transactions_clean["sales_qty"], 
    errors="coerce"
)

transactions_clean["sales_amount"] = pd.to_numeric(
    transactions_clean["sales_amount"], 
    errors="coerce"
)

# Standardising currency:

transactions_clean["currency"] = transactions_clean["currency"].str.upper()

# Removing exact duplicates after text cleanup:

transactions_clean = transactions_clean.drop_duplicates().reset_index(drop=True)

# Removing invalid rows:

transactions_clean = transactions_clean[
    (transactions_clean["order_date"].notna()) &
    (transactions_clean["sales_qty"] > 0) &
    (transactions_clean["sales_amount"] > 0)
].copy()

transactions_clean.head()

,product_code,customer_code,market_code,order_date,sales_qty,sales_amount,currency
0,Prod001,Cus001,Mark001,2017-10-10,100,41241,INR
2,Prod002,Cus003,Mark003,2018-04-06,1,875,INR
3,Prod002,Cus003,Mark003,2018-04-11,1,583,INR
4,Prod002,Cus004,Mark003,2018-06-18,6,7176,INR
5,Prod003,Cus005,Mark004,2017-11-20,59,500,USD


In [27]:
# Converting all sales amount values to INR:

USD_TO_INR = 96

transactions_clean["sales_amount_original"] = transactions_clean["sales_amount"]
transactions_clean["currency_original"] = transactions_clean["currency"]

transactions_clean["sales_amount_inr"] = np.where(
    transactions_clean["currency"] == "USD",
    transactions_clean["sales_amount"] * USD_TO_INR,
    transactions_clean["sales_amount"]
)

transactions_clean["currency"] = "INR"

transactions_clean.head()

,product_code,customer_code,market_code,order_date,sales_qty,sales_amount,currency,sales_amount_original,currency_original,sales_amount_inr
0,Prod001,Cus001,Mark001,2017-10-10,100,41241,INR,41241,INR,41241
2,Prod002,Cus003,Mark003,2018-04-06,1,875,INR,875,INR,875
3,Prod002,Cus003,Mark003,2018-04-11,1,583,INR,583,INR,583
4,Prod002,Cus004,Mark003,2018-06-18,6,7176,INR,7176,INR,7176
5,Prod003,Cus005,Mark004,2017-11-20,59,500,INR,500,USD,48000


### Checking relationships across tables

In [28]:
missing_products = set(transactions_clean["product_code"]) - set(products_clean["product_code"])
missing_customers = set(transactions_clean["customer_code"]) - set(customers_clean["customer_code"])
missing_markets = set(transactions_clean["market_code"]) - set(markets_clean["market_code"])
missing_dates = set(transactions_clean["order_date"]) - set(date_clean["date"])

print("Missing product codes:", missing_products)
print("Missing customer codes:", missing_customers)
print("Missing market codes:", missing_markets)
print("Missing dates:", missing_dates)

assert len(missing_products) == 0
assert len(missing_customers) == 0
assert len(missing_markets) == 0
assert len(missing_dates) == 0

Missing product codes: set()
Missing customer codes: set()
Missing market codes: set()
Missing dates: set()


### Final profiling after cleaning

In [29]:
for name, df in {
    "customers_clean": customers_clean,
    "date_clean": date_clean,
    "markets_clean": markets_clean,
    "products_clean": products_clean,
    "transactions_clean": transactions_clean,
}.items():
    basic_profile(df, name)


customers_clean
---------------
Shape: (38, 3)

Missing values:
customer_code    0
customer_name    0
customer_type    0
dtype: int64

Duplicate rows: 0

Data types:
customer_code    string[python]
customer_name    string[python]
customer_type    string[python]
dtype: object

date_clean
----------
Shape: (1126, 5)

Missing values:
date           0
cy_date        0
year           0
month_name     0
date_yy_mmm    0
dtype: int64

Duplicate rows: 0

Data types:
date           datetime64[ns]
cy_date         datetime64[s]
year                    int32
month_name             object
date_yy_mmm            object
dtype: object

markets_clean
-------------
Shape: (17, 3)

Missing values:
market_code     0
markets_name    0
zone            0
dtype: int64

Duplicate rows: 0

Data types:
market_code     string[python]
markets_name    string[python]
zone            string[python]
dtype: object

products_clean
--------------
Shape: (279, 3)

Missing values:
product_code    0
product_name    0
produ

### Exporting the files

In [30]:
customers_clean.to_csv(OUTPUT_DIR / "semicon_customers_clean.csv", index=False)
date_clean.to_csv(OUTPUT_DIR / "semicon_date_clean.csv", index=False)
markets_clean.to_csv(OUTPUT_DIR / "semicon_markets_clean.csv", index=False)
products_clean.to_csv(OUTPUT_DIR / "semicon_products_named_clean.csv", index=False)
transactions_clean.to_csv(OUTPUT_DIR / "semicon_transactions_clean.csv", index=False)

print(f"Cleaned CSV files exported to: {OUTPUT_DIR}")

Cleaned CSV files exported to: C:\Users\j111c\Semicon Sales Analysis Project\cleaned_csv


## End